# Self-learning homophonic carver — Colab T4 (disconnect-proof + monitored)
Checkpoints to Google Drive (auto-resumes after a disconnect), keeps the runtime awake, scores each round with the Nemotron judge, and pushes `status.json` to a `selflearn-status` branch so progress can be monitored remotely.
Runtime → Change runtime type → **T4 GPU**, then Run all.

In [ ]:
!nvidia-smi -L

In [ ]:
# --- Drive: checkpoints + status survive Colab disconnects ---
from google.colab import drive
drive.mount('/content/drive')
CKPT = '/content/drive/MyDrive/homophonic-carver'   # resume target
import os; os.makedirs(CKPT, exist_ok=True)

In [ ]:
# --- secrets (Colab: 🔑 left sidebar -> add as named secrets, or set inline) ---
from google.colab import userdata
def secret(name):
    try: return userdata.get(name)
    except Exception: return ''
os.environ['OPENROUTER_API_KEY'] = secret('OPENROUTER_API_KEY')  # for the LLM judge
os.environ['GITHUB_TOKEN']      = secret('GITHUB_TOKEN')         # for status push + private clone
os.environ['GITHUB_REPO']       = 'grundlagen/Lingua-Sound-Wave'

In [ ]:
# --- clone the repo (token used only if the repo is private) ---
BRANCH = 'claude/phrase-weave-multiword'
tok = os.environ.get('GITHUB_TOKEN','')
url = f"https://{tok+'@' if tok else ''}github.com/{os.environ['GITHUB_REPO']}.git"
!rm -rf repo && git clone --depth 1 -b $BRANCH $url repo
%cd repo/research/homophone-bench

In [ ]:
!apt-get -qq install -y espeak-ng >/dev/null
!pip -q install transformers trl datasets accelerate panphon wordfreq numpy
!python selflearn/reward.py   # sanity: prosody reward runs

In [ ]:
# --- keep-alive: reduce idle disconnects (also keep this browser tab open) ---
from IPython.display import Javascript, display
display(Javascript('''
  function _keepAlive(){ try{ document.querySelector('colab-connect-button')
    .shadowRoot.querySelector('#connect').click(); }catch(e){} }
  setInterval(_keepAlive, 60000);
'''))
print('keep-alive armed (best-effort; Colab Pro is more reliable for long runs)')

In [ ]:
# --- self-learning: checkpoints to Drive, resumes automatically, pushes status ---
%cd selflearn
!python train_selflearn.py --base Qwen/Qwen2.5-1.5B-Instruct \
    --rounds 6 --k 8 --keep_thresh 0.55 --ckpt_dir "$CKPT" --eval_llm
%cd ..

In [ ]:
# --- try the self-learned carver (loads from Drive checkpoint) ---
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
t = AutoTokenizer.from_pretrained(CKPT)
m = AutoModelForCausalLM.from_pretrained(CKPT, torch_dtype=torch.float16, device_map='auto')
for en in ['the little cat', 'my friend loves the sea', 'gentle rain falls']:
    p = f'Rewrite the English so it sounds the same when read aloud in French: {en}'
    ids = t.apply_chat_template([{'role':'user','content':p}], add_generation_prompt=True, return_tensors='pt').to(m.device)
    o = m.generate(ids, max_new_tokens=24, do_sample=False)
    print(en, '->', t.decode(o[0][ids.shape[1]:], skip_special_tokens=True).strip())

## Monitoring & resume
- **Resume:** if the runtime drops, just re-run the cells — training continues from the last Drive checkpoint (it reads `status.json` for the round).
- **Remote monitor:** with `GITHUB_TOKEN` set, each round pushes `selflearn/status.json` to the `selflearn-status` branch (round, kept count, mean reward, sample carves + Nemotron scores). Ask Claude to read that branch and advise/adjust.
- **Admin:** Claude changes the code on `claude/phrase-weave-multiword`; re-run the clone cell to pull and continue.